# Demo notebook
Run this code for each iteration/new data file.
## Overview
- Replace Sierra Item statused with new ones from map.


## Open and inspect file (Pandas)
Start by reading the file into a Pandas dataframe. If there is an encoding issue, use encoding="unicode_escape" parameter to read the file anyway.

In [ ]:
import pandas as pd
import csv
from pathlib import Path

filepath = Path("../iterations/test_run/source_data/loans/loans.tsv")
delimiter = "\t"
map = {
    "-": "",
    "n": "",
    "@": "",
    "w": "Withdrawn",
    "$": "Lost and paid",
    "m": "Missing",
    "o": "Missing"
    }

try:
    data = pd.read_csv(filepath, dtype=object, delimiter=delimiter, na_filter=False)     
    data["next_item_status"] = data["next_item_status"].map(lambda x: map.get(x.strip(), ""))
except UnicodeDecodeError:
    print("Escaping encoding error...")
    data = pd.read_csv(filepath, dtype=object, delimiter=delimiter, na_filter=False,encoding="unicode_escape")

# Show columns and some numbers
data.info()
data.head(2)

## Save to a new file in the same folder

In [ ]:
# Save with all fields quoted. This avoids issues reading preprocessed files in the migration tools.
data.to_csv(filepath[:-4] + "_prepped.tsv", sep="\t", index=False, quoting=csv.QUOTE_ALL)

## Open and inspect file (Polars)
Start by reading the file into a Polars dataframe (LazyFrame). If there is an encoding issue, use encoding="unicode_escape" parameter to read the file anyway.

In [ ]:
import polars as pl

filepath = Path("../iterations/test_run/source_data/loans/loans.tsv")
delimiter = "\t"
map = {
    "-": "",
    "n": "",
    "@": "",
    "w": "Withdrawn",
    "$": "Lost and paid",
    "m": "Missing",
    "o": "Missing"
    }

try:
    data = pl.scan_csv(filepath, separator=delimiter, infer_schema_length=False) # Infer all columns as string types (you can cast them later if needed)
    data = data.with_columns(
        pl.col("next_item_status").replace(map)
    )
except UnicodeDecodeError:
    print("Escaping encoding error...")
    data = pl.scan_csv(filepath, separator=delimiter, infer_schema_length=False, encoding="unicode_escape")
# Show columns and some numbers

data.filter(
    pl.col("next_item_status").is_in(["Lost and paid", "Missing", "Withdrawn"])
).collect().head(2) # you need to call collect() to execute the query plan on a LazyFrame object

In [ ]:
# Save with all fields quoted. This avoids issues reading preprocessed files in the migration tools.
data.collect().write_csv(filepath.parent.joinpath(filepath.stem + "_prepped_polars.tsv"), separator="\t", quote_style="always")